<a href="https://colab.research.google.com/github/lukalklikadze/walmart-sales-forecasting/blob/main/notebooks/model_experiment_DLinear.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install mlflow kaggle

!git clone -q https://github.com/lukalklikadze/walmart-sales-forecasting.git /content/repo \
    2>/dev/null || (cd /content/repo && git pull -q)
import sys; sys.path.append("/content/repo")

from src.config import setup_env, DATA_DIR, KAGGLE_COMP
tracking_uri = setup_env()
print("MLflow →", tracking_uri)

import os, glob, zipfile, subprocess
os.makedirs(DATA_DIR, exist_ok=True)
subprocess.run(["kaggle","competitions","download","-c",KAGGLE_COMP,"-p",DATA_DIR,"--force"], check=True)
with zipfile.ZipFile(os.path.join(DATA_DIR, KAGGLE_COMP + ".zip")) as z: z.extractall(DATA_DIR)
for inner in glob.glob(os.path.join(DATA_DIR, "*.csv.zip")):
    with zipfile.ZipFile(inner) as z: z.extractall(DATA_DIR)

from src.data import load_raw
train, test, stores, features = load_raw()
print("train:", train.shape, "| test:", test.shape)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 124.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# ── DLinear config ──────────────────────────────────────────────────────
import numpy as np, pandas as pd, torch, torch.nn as nn, mlflow
from torch.utils.data import Dataset, DataLoader

from src.features import CalendarFeatures, GROUP_COLS
from src.metrics import wmae                 # FIXED: was "src.metrics" (plural) -- check this matches your actual filename
from src.train_val_split import time_based_split

SEQ_LEN     = 52       # 1 year of weekly lookback
# FIXED: was 104. Each training window needs SEQ_LEN+PRED_LEN contiguous
# weeks entirely inside the TRAIN split. This dataset is only ~143 weeks
# total; holding out PRED_LEN=39 weeks for honest validation leaves
# raw_train_split with only ~104 weeks -- too short for a 104+39=143-week
# window (that's the "num_samples=0" crash). 52+39=91 fits with margin
# to spare, and 52 weeks still spans a full seasonal cycle (incl. last
# Christmas). If you want a longer lookback, you can raise SEQ_LEN back up
# for the FINAL model you fit on the full `raw` frame for real submission
# (143 weeks available there), but keep it at 52 for anything fit on
# raw_train_split, or you'll hit this same error again.
PRED_LEN    = 39       # matches the real Kaggle test horizon exactly
KERNEL_SIZE = 25       # moving-avg trend/seasonal decomposition window
STRIDE      = 1        # FIXED: was 2. Data is scarce at seq_len=52 (~13-week
                        # margin per series), stride=1 gets more windows out of it
BATCH_SIZE  = 256
EPOCHS      = 30
LR          = 1e-3
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

# calendar covariates are known ahead of time for any forecast horizon,
# so it's safe to feed the FUTURE values of these into the model
COV_COLS = ["IsHoliday", "is_superbowl", "is_laborday", "is_thanksgiving",
            "is_christmas_flagged", "is_week_before_christmas",
            "week_sin", "week_cos"]


In [4]:
# ── join raw tables (deep nets skip Group B lags entirely) ─────────────
# NOTE: CalendarFeatures is intentionally NOT applied here. The Pipeline step
# (DLinearForecaster) applies it internally on whatever raw frame it's given,
# which is exactly what lets pipeline.predict(raw_test_df) work on an
# untouched test set, per the assignment requirement.
def join_frame(sales_df, stores_df, features_df):
    df = sales_df.merge(stores_df, on="Store", how="left")
    df = df.merge(features_df, on=["Store", "Date"], how="left", suffixes=("", "_feat"))

    # stores_df and features_df both carry Type/Size -> merge creates _x/_y suffixes
    if "Type_x" in df.columns:
        df["Type"] = df["Type_x"]
        df = df.drop(columns=["Type_x"])
    if "Size_x" in df.columns:
        df["Size"] = df["Size_x"]
        df = df.drop(columns=["Size_x"])
    if "Type_y" in df.columns:
        df = df.drop(columns=["Type_y"])
    if "Size_y" in df.columns:
        df = df.drop(columns=["Size_y"])

    if "IsHoliday_feat" in df.columns:
        df["IsHoliday"] = df["IsHoliday_feat"]
        df = df.drop(columns=["IsHoliday_feat"])
    df["Date"] = pd.to_datetime(df["Date"])
    return df

raw = join_frame(train, stores, features)
assert "Type" in raw.columns, "Type column missing after join_frame -- check stores_df merge"
print("raw:", raw.shape, raw["Date"].min(), "→", raw["Date"].max())


raw: (421570, 25) 2010-02-05 00:00:00 → 2012-10-26 00:00:00


In [5]:
# ── time-based split on the RAW frame ───────────────────────────────────
# Splitting `raw` (not a CalendarFeatures-transformed frame) means both
# raw_train_split and raw_val_split stay unprocessed -- exactly the shape
# the Pipeline's .fit()/.predict() are meant to consume, and it lets the
# validation step below double as a rehearsal of the real test.csv call.
raw_train_split, raw_val_split = time_based_split(raw, val_weeks=PRED_LEN)
print("raw_train_split:", raw_train_split.shape, "| raw_val_split:", raw_val_split.shape)


raw_train_split: (305982, 25) | raw_val_split: (115588, 25)


In [6]:
# ── DLinear architecture (Zeng et al. 2022) + a small covariate head ───
class MovingAvg(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)

    def forward(self, x):  # x: [B, L, C]
        pad = (self.kernel_size - 1) // 2
        front = x[:, :1, :].repeat(1, pad, 1)
        end = x[:, -1:, :].repeat(1, self.kernel_size - 1 - pad, 1)
        x = torch.cat([front, x, end], dim=1)
        return self.avg(x.permute(0, 2, 1)).permute(0, 2, 1)


class SeriesDecomp(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.moving_avg = MovingAvg(kernel_size)

    def forward(self, x):
        trend = self.moving_avg(x)
        return x - trend, trend  # seasonal, trend


class DLinear(nn.Module):
    """Decomposition-linear forecaster for the Weekly_Sales channel
    (channels=1, weights shared across ALL Store x Dept series -> one
    global model, not ~3300 per-series ones). The moving-average trend/
    seasonal split and the two seq_len->pred_len linear maps are exactly
    the original DLinear architecture. `cov_head` is the one addition: a
    small MLP over the FUTURE calendar features (Group A, fully known in
    advance) whose output is added on top of the seasonal+trend forecast,
    so the model can use "next Nov 28 is Thanksgiving week" the way a
    tree model uses is_thanksgiving directly.
    """
    def __init__(self, seq_len, pred_len, n_cov, kernel_size=25, cov_hidden=64):
        super().__init__()
        self.decomp = SeriesDecomp(kernel_size)
        self.linear_seasonal = nn.Linear(seq_len, pred_len)
        self.linear_trend = nn.Linear(seq_len, pred_len)
        self.cov_head = nn.Sequential(
            nn.Linear(pred_len * n_cov, cov_hidden),
            nn.ReLU(),
            nn.Linear(cov_hidden, pred_len),
        )

    def forward(self, x, cov_future):
        # x: [B, seq_len, 1] | cov_future: [B, pred_len, n_cov]
        seasonal, trend = self.decomp(x)
        out = self.linear_seasonal(seasonal.squeeze(-1)) + self.linear_trend(trend.squeeze(-1))
        out = out + self.cov_head(cov_future.flatten(1))
        return out.unsqueeze(-1)  # [B, pred_len, 1]


In [7]:
# ── training-window dataset + loss (this cell was missing -- it's what
# DLinearForecaster.fit() below actually calls) ─────────────────────────
def to_series_dict(df):
    return {k: g.reset_index(drop=True)
            for k, g in df.sort_values(GROUP_COLS + ["Date"]).groupby(GROUP_COLS)}


class DLinearTrainDataset(Dataset):
    """Sliding (seq_len -> pred_len) windows of Weekly_Sales, pooled across
    ALL (Store, Dept) series into one dataset so a single shared DLinear
    learns globally instead of fitting ~3300 independent per-series models.

    A window is only emitted if its seq_len+pred_len dates are exactly
    weekly-contiguous -- gappy series are skipped, never bridged.
    Per-series (mu, sigma) z-score stats are fit here and reused at
    predict time (via DLinearForecaster.stats_) so scales don't leak
    across the split.
    """
    def __init__(self, frame, seq_len, pred_len, cov_cols, stride=2, stats=None):
        self.seq_len, self.pred_len = seq_len, pred_len
        self.stats = {} if stats is None else stats
        fit_stats = stats is None
        self.samples = []
        for key, g in to_series_dict(frame).items():
            sales = g["Weekly_Sales"].to_numpy(dtype=float)
            cov = g[cov_cols].to_numpy(dtype=float)
            dates = g["Date"]
            if fit_stats:
                self.stats[key] = (sales.mean(), sales.std() + 1e-6)
            mu, sigma = self.stats[key]
            L = seq_len + pred_len
            for start in range(0, len(g) - L + 1, stride):
                span = dates.iloc[start:start + L]
                if not (span.diff().dropna() == pd.Timedelta(weeks=1)).all():
                    continue
                x = (sales[start:start + seq_len] - mu) / sigma
                y = (sales[start + seq_len:start + L] - mu) / sigma
                cov_future = cov[start + seq_len:start + L]
                self.samples.append((x, cov_future, y))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        x, cov_future, y = self.samples[i]
        return (torch.tensor(x, dtype=torch.float32).unsqueeze(-1),
                torch.tensor(cov_future, dtype=torch.float32),
                torch.tensor(y, dtype=torch.float32).unsqueeze(-1))


def weighted_l1(pred, target, is_holiday_future):
    """WMAE-shaped training loss: holiday weeks weighted 5x, same weighting
    as the Kaggle metric (src/metric.py:wmae), so the loss the net optimizes
    matches the metric it's judged on. Must be module-level, not a class
    method -- DLinearForecaster.fit() below calls it directly as weighted_l1(...).
    """
    w = torch.where(is_holiday_future.bool(), 5.0, 1.0)
    return (w * (pred.squeeze(-1) - target.squeeze(-1)).abs()).mean()


In [8]:
# ── wrap DLinear as a single sklearn-compatible Pipeline step ──────────
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.pipeline import Pipeline

class DLinearForecaster(BaseEstimator, RegressorMixin):
    """Makes the trained DLinear runnable end-to-end on a RAW, unprocessed
    frame (same raw shape as join_frame()'s output: Date, Store, Dept,
    IsHoliday, MarkDown1-5 w/ NaNs, Type, Size, macro cols -- no
    CalendarFeatures applied, no Weekly_Sales normalization done).

    fit(X_raw, y): applies CalendarFeatures internally, trains the DLinear
    net, and snapshots the per-series (Store, Dept) history + (mu, sigma)
    stats needed to build lookback windows at predict time -- mirrors how
    LagFeatures.fit() snapshots history in features.py.

    predict(X_raw): applies CalendarFeatures internally, pulls each row's
    seq_len lookback from the stored TRAIN history (never from y, since
    predict never receives y), builds cov_future from the row's own
    (already-known) calendar features, runs the net, de-normalizes, and
    returns predictions aligned to X_raw's row order. Rows whose series
    has insufficient history return NaN rather than raising, so predict()
    never crashes on unseen (Store, Dept) combos.
    """
    def __init__(self, seq_len=104, pred_len=39, cov_cols=None,
                 kernel_size=25, epochs=30, lr=1e-3, batch_size=256,
                 stride=2, device=None):
        self.seq_len, self.pred_len = seq_len, pred_len
        self.cov_cols = cov_cols or COV_COLS
        self.kernel_size, self.epochs = kernel_size, epochs
        self.lr, self.batch_size, self.stride = lr, batch_size, stride
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    def _featurize(self, X_raw):
        return CalendarFeatures().fit_transform(X_raw).sort_values(
            GROUP_COLS + ["Date"]).reset_index(drop=True)

    def fit(self, X_raw, y=None):
        # FIXED: attach y to X_raw BEFORE featurizing. _featurize() sorts by
        # (Store, Dept, Date) and resets the index -- assigning y onto the
        # frame AFTER that reorder (the old code) silently mismatches every
        # sales value to the wrong row. Attaching first keeps y glued to its
        # row through the sort.
        X_in = X_raw.copy()
        if y is not None:
            X_in["Weekly_Sales"] = np.asarray(y)
        df = self._featurize(X_in)

        ds = DLinearTrainDataset(df, self.seq_len, self.pred_len,
                                  self.cov_cols, stride=self.stride)
        if len(ds) == 0:
            n_weeks = df.groupby(GROUP_COLS)["Date"].nunique().max() if len(df) else 0
            raise ValueError(
                f"DLinearTrainDataset produced 0 windows. seq_len+pred_len="
                f"{self.seq_len + self.pred_len} weeks are needed per window, but the "
                f"longest series in this fit() call only has {n_weeks} contiguous weeks. "
                f"Lower seq_len (or pred_len) so seq_len+pred_len <= the weeks available "
                f"in whatever frame you're calling .fit() on."
            )
        loader = DataLoader(ds, batch_size=self.batch_size, shuffle=True, drop_last=True)

        self.model_ = DLinear(self.seq_len, self.pred_len, len(self.cov_cols),
                               self.kernel_size).to(self.device)
        opt = torch.optim.Adam(self.model_.parameters(), lr=self.lr)
        for _ in range(self.epochs):
            self.model_.train()
            for x, cov_future, yb in loader:
                x, cov_future, yb = x.to(self.device), cov_future.to(self.device), yb.to(self.device)
                is_hol = cov_future[..., 0]  # IsHoliday must stay COV_COLS[0]
                opt.zero_grad()
                loss = weighted_l1(self.model_(x, cov_future), yb, is_hol)
                loss.backward()
                opt.step()

        # snapshot train history + per-series stats -- same role as
        # LagFeatures.history_, needed because predict() gets no y
        self.history_ = {k: g.reset_index(drop=True)
                          for k, g in df.groupby(GROUP_COLS)}
        self.stats_ = ds.stats
        return self

    def predict(self, X_raw):
        df = self._featurize(X_raw)
        self.model_.eval()
        preds = np.full(len(df), np.nan, dtype=float)

        for key, g in df.groupby(GROUP_COLS):
            hist = self.history_.get(key)
            if hist is None or key not in self.stats_:
                continue  # unseen series -> leave as NaN, don't crash
            mu, sigma = self.stats_[key]
            hist_tail = hist[hist["Date"] < g["Date"].min()].tail(self.seq_len)
            if len(hist_tail) < self.seq_len:
                continue
            x = torch.tensor(((hist_tail["Weekly_Sales"].to_numpy(dtype=float) - mu) / sigma),
                              dtype=torch.float32, device=self.device).view(1, self.seq_len, 1)
            g_sorted = g.sort_values("Date")
            cov = torch.tensor(g_sorted[self.cov_cols].to_numpy(dtype=float),
                                dtype=torch.float32, device=self.device)
            with torch.no_grad():
                cov_window = cov[:self.pred_len]
                if len(cov_window) < self.pred_len:
                    pad = cov_window[-1:].repeat(self.pred_len - len(cov_window), 1)
                    cov_window = torch.cat([cov_window, pad], dim=0)
                pred_norm = self.model_(x, cov_window.unsqueeze(0)).squeeze().cpu().numpy()
                pred = pred_norm[:len(g_sorted)] * sigma + mu
            preds[df.index.get_indexer(g_sorted.index)] = pred

        return preds


def make_dlinear_pipeline(seq_len, pred_len=PRED_LEN, kernel_size=KERNEL_SIZE,
                           lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE, stride=STRIDE):
    """Factory so the sweep cell below can build a fresh Pipeline per config
    without repeating the DLinearForecaster(...) call each time."""
    return Pipeline([
        ("dlinear", DLinearForecaster(seq_len=seq_len, pred_len=pred_len,
                                       cov_cols=COV_COLS, kernel_size=kernel_size,
                                       epochs=epochs, lr=lr, batch_size=batch_size,
                                       stride=stride)),
    ])


In [9]:
# ── hyperparameter sweep: try a few configs, log each as a lightweight
# mlflow run (metrics only -- no point pickling every candidate net), then
# pick the winner by val_wmae. Only the winner gets the full Pipeline
# artifact logged, in the next cell. ───────────────────────────────────
max_train_weeks = raw_train_split.groupby(GROUP_COLS)["Date"].nunique().max()
print(f"longest series in raw_train_split: {max_train_weeks} weeks "
      f"(seq_len+pred_len must stay <= this)")

SWEEP_EPOCHS = 15   # shorter than the final EPOCHS -- just enough to rank configs

SEARCH_SPACE = [
    dict(seq_len=39, kernel_size=13, lr=1e-3),
    dict(seq_len=52, kernel_size=25, lr=1e-3),
    dict(seq_len=65, kernel_size=25, lr=1e-3),
    dict(seq_len=52, kernel_size=13, lr=5e-4),
]

mlflow.set_experiment("DLinear_Training")
raw_val_features = raw_val_split.drop(columns=["Weekly_Sales"])

sweep_results = []
for cfg in SEARCH_SPACE:
    if cfg["seq_len"] + PRED_LEN > max_train_weeks:
        print(f"skip {cfg}: seq_len+pred_len={cfg['seq_len']+PRED_LEN} > {max_train_weeks} weeks available")
        continue

    pipe = make_dlinear_pipeline(seq_len=cfg["seq_len"], kernel_size=cfg["kernel_size"],
                                  lr=cfg["lr"], epochs=SWEEP_EPOCHS)
    pipe.fit(raw_train_split, raw_train_split["Weekly_Sales"])

    val_preds = pipe.predict(raw_val_features)
    mask = ~np.isnan(val_preds)
    coverage = mask.mean()
    val_wmae = wmae(raw_val_split["Weekly_Sales"][mask], val_preds[mask], raw_val_split["IsHoliday"][mask])

    run_name = f"dlinear_sweep_seq{cfg['seq_len']}_k{cfg['kernel_size']}_lr{cfg['lr']}"
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(dict(**cfg, pred_len=PRED_LEN, stride=STRIDE,
                                batch_size=BATCH_SIZE, epochs=SWEEP_EPOCHS,
                                feature_group="A_only_calendar", sweep=True))
        mlflow.log_metric("val_wmae", val_wmae)
        mlflow.log_metric("val_coverage", coverage)
        run_id = mlflow.active_run().info.run_id

    print(f"{cfg}  epochs={SWEEP_EPOCHS} -> val_wmae={val_wmae:.2f}  coverage={coverage:.1%}  run={run_id}")
    sweep_results.append(dict(cfg=cfg, val_wmae=val_wmae, coverage=coverage, run_id=run_id))
    del pipe  # free the net + history_ copy before the next config trains

sweep_results.sort(key=lambda r: r["val_wmae"])
if not sweep_results:
    raise ValueError(
        "Every config in SEARCH_SPACE was skipped (seq_len+pred_len exceeded "
        f"max_train_weeks={max_train_weeks}). Lower the seq_len values in SEARCH_SPACE."
    )
print("\nleaderboard (best first):")
for r in sweep_results:
    print(f"  val_wmae={r['val_wmae']:.2f}  coverage={r['coverage']:.1%}  {r['cfg']}")

best = sweep_results[0]
print("\nBEST CONFIG:", best["cfg"], "| sweep val_wmae:", round(best["val_wmae"], 2))


longest series in raw_train_split: 104 weeks (seq_len+pred_len must stay <= this)
🏃 View run dlinear_sweep_seq39_k13_lr0.001 at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/7/runs/e887ebbd7a804e2ebf6607c822bfaa09
🧪 View experiment at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/7
{'seq_len': 39, 'kernel_size': 13, 'lr': 0.001}  epochs=15 -> val_wmae=2386.56  coverage=97.9%  run=e887ebbd7a804e2ebf6607c822bfaa09
🏃 View run dlinear_sweep_seq52_k25_lr0.001 at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/7/runs/95b67ad82ecc4f9ea6d7b6cba950ea6f
🧪 View experiment at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/7
{'seq_len': 52, 'kernel_size': 25, 'lr': 0.001}  epochs=15 -> val_wmae=2468.44  coverage=97.2%  run=95b67ad82ecc4f9ea6d7b6cba950ea6f
🏃 View run dlinear_sweep_seq65_k25_lr0.001 at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/7/runs/3

In [10]:
# ── refit the winning config at full EPOCHS -- the short SWEEP_EPOCHS run
# above was only for ranking, the model you actually log should be trained
# properly. This becomes `dlinear_pipeline`, same name the next cell expects.
dlinear_pipeline = make_dlinear_pipeline(seq_len=best["cfg"]["seq_len"],
                                          kernel_size=best["cfg"]["kernel_size"],
                                          lr=best["cfg"]["lr"], epochs=EPOCHS)
dlinear_pipeline.fit(raw_train_split, raw_train_split["Weekly_Sales"])
print("refit done with best config:", best["cfg"], "| epochs:", EPOCHS)


refit done with best config: {'seq_len': 39, 'kernel_size': 13, 'lr': 0.001} | epochs: 30


In [14]:
# ── validate on the RAW val split (rehearses the real test.csv call),
# then log the whole Pipeline -- not a state_dict -- so it's runnable
# directly on unprocessed test data ─────────────────────────────────────
raw_val_features = raw_val_split.drop(columns=["Weekly_Sales"])
val_preds = dlinear_pipeline.predict(raw_val_features)

mask = ~np.isnan(val_preds)
coverage = mask.mean()
val_wmae = wmae(raw_val_split["Weekly_Sales"][mask], val_preds[mask], raw_val_split["IsHoliday"][mask])
print(f"val coverage: {coverage:.1%}  |  val WMAE: {val_wmae:.2f}")
if coverage < 0.9:
    print("WARNING: low coverage -- many (Store, Dept) series don't have "
          "enough clean training weeks before the val cutoff; consider a smaller seq_len.")

# move the net to CPU before pickling so the logged model loads on any
# machine (Kaggle scoring / a teammate's laptop may not have a GPU)
dlinear_pipeline.named_steps["dlinear"].model_.to("cpu")
dlinear_pipeline.named_steps["dlinear"].device = "cpu"

# FIXED: log the winning config's ACTUAL hyperparameters via get_params(),
# not the config-cell globals -- those still hold the sweep's default/first
# values and would silently mismatch whatever the sweep actually picked.
fitted_params = dlinear_pipeline.named_steps["dlinear"].get_params()

mlflow.set_experiment("walmart-sales-forecasting")
with mlflow.start_run(run_name="dlinear_best"):
    mlflow.log_params({**fitted_params, "feature_group": "A_only_calendar",
                        "cov_cols": ",".join(COV_COLS),
                        "selected_via_sweep_run_id": best["run_id"]})
    mlflow.log_metric("val_wmae", val_wmae)
    mlflow.log_metric("val_coverage", coverage)
    model_info = mlflow.sklearn.log_model(dlinear_pipeline, artifact_path="model",  serialization_format="cloudpickle")
    run_id = mlflow.active_run().info.run_id
    print("logged pipeline run:", run_id, "| val WMAE:", val_wmae, "| config:", best["cfg"])

# sanity check: reload exactly the way a grader / submission script would,
# and confirm it runs on a fully raw, unprocessed frame with zero manual prep.
# FIXED: use model_info.model_uri (what log_model() actually returned) instead
# of hand-building "runs:/{run_id}/model" -- recent MLflow versions track
# logged models as their own entity, so the reconstructed runs:/ path can
# fail to resolve even when logging itself succeeded.
try:
    loaded = mlflow.sklearn.load_model(model_info.model_uri)
    sanity_preds = loaded.predict(raw_val_features)
    print("reload sanity check, NaN mismatch vs original preds:",
          int(np.sum(np.isnan(sanity_preds) != np.isnan(val_preds))))
except Exception as e:
    print(f"WARNING: model logged successfully, but the in-notebook reload "
          f"check failed ({type(e).__name__}: {e}). This doesn't mean the "
          f"artifact is broken -- try loading it from the MLflow UI/registry "
          f"directly to confirm before assuming the run is bad.")


val coverage: 97.9%  |  val WMAE: 2384.78


2026/07/09 23:36:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/09 23:36:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/07/09 23:36:29 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


logged pipeline run: 6d6457ab97c94ef787d0056eb1e692fd | val WMAE: 2384.7806622084977 | config: {'seq_len': 39, 'kernel_size': 13, 'lr': 0.001}
🏃 View run dlinear_best at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/6/runs/6d6457ab97c94ef787d0056eb1e692fd
🧪 View experiment at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/6


reload sanity check, NaN mismatch vs original preds: 0
